<a href="https://colab.research.google.com/github/Asheesh1272/Asheesh127/blob/main/alziemer_classification_cnn_98_accuracy1_under_the_guidence_IITB_CSE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import tqdm
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import lr_scheduler
import torch.backends.cudnn as cudnn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import cv2
import torchvision
from torchvision import datasets, models, transforms
from torchvision.transforms import v2
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import time
import os
import PIL
from PIL import Image
cudnn.benchmark = True

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Using {device} device")


In [ ]:
BASE_DIR = "/content/Alzheimer MRI Disease Classification Dataset/Data/"
disease_label_from_category = {
    0: "Mild Demented",
    1: "Moderate Demented",
    2: "Non Demented",
    3: "Very Mild Demented",
}

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("borhanitrash/alzheimer-mri-disease-classification-dataset")

print("Path to dataset files:", path)

In [ ]:
!unzip /content/archive.zip -d /content/

In [ ]:
df = pd.read_parquet(f"{path}/Alzheimer MRI Disease Classification Dataset/Data/train-00000-of-00001-c08a401c53fe5312.parquet", engine="pyarrow")
display(df.head())

In [ ]:
test = pd.read_parquet(f"{path}/Alzheimer MRI Disease Classification Dataset/Data/test-00000-of-00001-44110b9df98c5585.parquet", engine="pyarrow")

In [ ]:
def dict_to_image(image_dict):
    if isinstance(image_dict, dict) and 'bytes' in image_dict:
        byte_string = image_dict['bytes']
        nparr = np.frombuffer(byte_string, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_GRAYSCALE)
        return img
    else:
        raise TypeError(f"Expected dictionary with 'bytes' key, got {type(image_dict)}")

In [ ]:
df['img_arr'] = df['image'].apply(dict_to_image)
df.drop("image", axis=1, inplace=True)
df.head()

In [ ]:
test['img_arr'] = test['image'].apply(dict_to_image)
test.drop("image", axis=1, inplace=True)

In [ ]:
# Check we can actually render the image and that it looks reasonable
fig, ax = plt.subplots(2, 3, figsize=(15, 5))
axs = ax.flatten()
for axes in axs:
    rand = np.random.randint(0, len(df))
    axes.imshow(df.iloc[rand]['img_arr'], cmap="gray")
    axes.set_title(disease_label_from_category[df.iloc[rand]['label']])
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(np.arange(0, 4, 1), df['label'].value_counts().sort_index())
plt.ylabel("Number of Images")
plt.xticks(np.arange(0, 4, 1), labels=[disease_label_from_category[i] for i in range(4)])
plt.show()
print(f"Total samples in training data = {len(df)}")

In [ ]:
N_CLASSES = df['label'].nunique()


In [ ]:
# Use a torch dataset/dataloader to handle feeding our data in the model
class ImageDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        image = self.dataframe.iloc[idx]["img_arr"]
        label = self.dataframe.iloc[idx]["label"]

        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(label, dtype=torch.long)
        return image, label

In [ ]:
class BaselineCNN(nn.Module):

    # layer as the output with the output shape being the number of classes
    def __init__(self):
        super(BaselineCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.batchnorm1 = nn.BatchNorm2d(num_features=32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 32 * 32, 128)
        self.out = nn.Linear(128, N_CLASSES)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        x = self.batchnorm1(x)
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = self.out(x)
        return x

In [ ]:
learning_rate = 0.001
NEPOCHS = 10
batch_size = 32

In [ ]:
# Create dataset and dataloader
train_dataset = ImageDataset(df)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
def train_model(model, loader, optimizer, num_epochs=NEPOCHS):

    criterion = nn.CrossEntropyLoss()

    # Training loop
    train_losses = []
    for epoch in tqdm.tqdm(range(num_epochs), total=num_epochs):
        running_loss = 0.0
        for i, data in enumerate(loader, 0):
            inputs, labels = data[0].to(device), data[1].to(device)

            optimizer.zero_grad()
            # forward + backward + optimize
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            # print statistics
            running_loss += loss.item()
        epoch_loss = running_loss / len(loader)
        train_losses.append(epoch_loss)

    print('Finished Training')
    return model, train_losses

In [ ]:
model = BaselineCNN().to(device)
optimizer = optim.AdamW(model.parameters(), lr=learning_rate)
model, train_losses = train_model(model, train_loader, optimizer)

In [ ]:
plt.plot(np.arange(1, 11), train_losses)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

In [ ]:
def predict(m, dl, device):
    m.eval()  # Set model to evaluation mode
    predictions = []
    true_labels = []

    with torch.no_grad():
        for images, labels in dl:
            images = images.to(device)
            outputs = m(images)
            _, preds = torch.max(outputs, 1)
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    return predictions, true_labels

def result_summary(predictions, true_labels):
    # Accuracy
    accuracy = accuracy_score(true_labels, predictions)
    print(f'Accuracy: {accuracy:.4f}')

    # Confusion Matrix
    conf_matrix = confusion_matrix(true_labels, predictions)
    print('Confusion Matrix:')
    print(conf_matrix)

In [ ]:
predictions, true_labels = predict(model, train_loader, device)
result_summary(predictions, true_labels)

In [ ]:

test_dataset = ImageDataset(test)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
predictions_test, test_labels = predict(model, test_loader, device)

result_summary(predictions_test, test_labels)


In [ ]:
class ImageDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        image = self.dataframe.iloc[idx]["img_arr"]
        label = self.dataframe.iloc[idx]["label"]

        if self.transform:
            image = image.astype(np.uint8)
            image = self.transform(image)

        label = torch.tensor(label, dtype=torch.long)
        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)

        return image, label

In [ ]:
transforms = v2.Compose([
    v2.RandomHorizontalFlip(p=0.3),
    v2.RandomVerticalFlip(p=0.3),
    v2.GaussianBlur(kernel_size=3),
    v2.RandomRotation(degrees=(-45, 45)),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485], std=[0.229]),
])

In [ ]:
train_dataset = ImageDataset(df, transforms) # Use df instead of train_df
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
# The only thing I have changed is adding transforms (+5 epochs of train time)
model_aug = BaselineCNN().to(device)
optimizer = optim.AdamW(model_aug.parameters(), lr=learning_rate)
model_aug, train_losses_aug = train_model(model_aug, train_loader, optimizer, num_epochs=15)

In [ ]:
plt.figure(figsize=(9,4))
plt.plot(np.arange(1, 15+1), train_losses_aug)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid()
plt.show()

In [ ]:
# Do not apply augmentation when generating predictions for evaluation of model performance
train_dataset_notf = ImageDataset(df, None)
train_loader_notf = DataLoader(train_dataset_notf, batch_size=batch_size, shuffle=True)

train_predictions, train_labels = predict(model_aug, train_loader_notf, device)
test_predictions, test_labels = predict(model_aug, test_loader, device)

In [ ]:
result_summary(train_predictions, train_labels)


In [ ]:
result_summary(test_predictions, test_labels)


In [ ]:
# I've kept the cell here in case it is a useful reference for someone later
"""
model_tf = models.resnet34(weights='IMAGENET1K_V1')

for param in model_tf.parameters(): # Freeze all but the last layer of the network
    param.requires_grad = False

num_ftrs = model_tf.fc.in_features
model_tf.fc = nn.Linear(num_ftrs, N_CLASSES)
model_tf = model_tf.to(device)

# Note ResNet needs 3 channels! Our grayscale image only has 3, let's paste them
# to make the image "RGB" (duplicate grayscale channel 3 times)
class GrayscaleToRGB(object):
    def __call__(self, image):
        return np.array(Image.fromarray(image).convert("RGB"))

transformsRGB = v2.Compose([
    GrayscaleToRGB(),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.5),
    v2.GaussianBlur(kernel_size=3),
    v2.RandomRotation(degrees=(-45, 45)),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.485, 0.485], std=[0.229, 0.229, 0.229]),
])

train_dataset_rgb = ImageDataset(df, transformsRGB, add_dimension=False)
train_loader_rgb = DataLoader(train_dataset_rgb, batch_size=batch_size, shuffle=True)

optimizer = optim.AdamW(model_tf.fc.parameters(), lr=learning_rate) # todo: Add momentum?
model3, train_losses3 = train_model(model_tf, train_loader_rgb, optimizer, num_epochs=20)

plt.plot(np.arange(1, 20+1), train_losses3)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()
"""

In [ ]:
# When we make predictions we now need to add the dummy dimensions back on
"""
eval_transforms = transformsRGB = v2.Compose([
    GrayscaleToRGB()
])

train_data_eval = ImageDataset(df, eval_transforms, add_dimension=False)
train_load_eval = DataLoader(train_data_eval, batch_size=batch_size, shuffle=True)
val_data_eval = ImageDataset(val, eval_transforms, add_dimension=False)
val_load_eval = DataLoader(val_data_eval, batch_size=batch_size, shuffle=True)

train_predictions, train_labels = predict(model3, train_load_eval, device)
val_predictions, val_labels = predict(model3, val_load_eval, device)

result_summary(train_predictions, train_labels)
result_summary(val_predictions, val_labels)
"""

In [ ]:
class TunedCNN(nn.Module):
    def __init__(self):
        super(TunedCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.batchnorm1 = nn.BatchNorm2d(num_features=32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 32 * 32, 128)
        self.drop1 = nn.Dropout(p=0.2)
        self.out = nn.Linear(128, N_CLASSES)

    def forward(self, x):
        x = F.mish(self.conv1(x))
        x = self.pool1(x)
        x = self.batchnorm1(x)
        x = F.mish(self.conv2(x))
        x = self.pool2(x)
        x = self.flatten(x)
        x = self.fc1(x)
        leaky = nn.LeakyReLU(0.01)
        x = leaky(x)
        x = self.drop1(x)
        x = self.out(x)
        return x

In [ ]:
class TunedCNN(nn.Module):
    def __init__(self):
        super(TunedCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.batchnorm1 = nn.BatchNorm2d(num_features=32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 32 * 32, 128)
        self.drop1 = nn.Dropout(p=0.2)
        self.out = nn.Linear(128, N_CLASSES)

    def forward(self, x):
        x = F.mish(self.conv1(x))
        x = self.pool1(x)
        x = self.batchnorm1(x)
        x = F.mish(self.conv2(x))
        x = self.pool2(x)
        x = self.flatten(x)
        x = self.fc1(x)
        leaky = nn.LeakyReLU(0.01)
        x = leaky(x)
        x = self.drop1(x)
        x = self.out(x)
        return x

model_tune = TunedCNN().to(device)
optimizer = optim.AdamW(model_tune.parameters(), lr=learning_rate)
model_tune, train_losses_tune = train_model(model_tune, train_loader, optimizer, num_epochs=20)

In [ ]:
plt.figure(figsize=(9,4))
plt.plot(np.arange(1, 20+1), train_losses_tune)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid()
plt.show()


In [ ]:
train_predictions, train_labels = predict(model_tune, train_loader_notf, device)
test_predictions, test_labels = predict(model_tune, test_loader, device)

result_summary(train_predictions, train_labels)
print()
result_summary(test_predictions, test_labels)

In [ ]:
torch.save(model_tune.state_dict(), "/content/alzheimer_cnn_model.pth")

In [ ]:
import pandas as pd
import numpy as np
import cv2 # OpenCV for image handling
import os

# --- Configuration ---
PARQUET_FILE_PATH = "/kaggle/input/alzheimer-mri-disease-classification-dataset/Alzheimer MRI Disease Classification Dataset/Data/train-00000-of-00001-c08a401c53fe5312.parquet" # Or your test file path
OUTPUT_FOLDER = "/kaggle/working/test_images/" # Where to save the JPGs (Kaggle's writable directory)
NUM_IMAGES_PER_CLASS = 2
CLASS_NAMES = ['Mild_Dementia', 'Moderate_Dementia', 'Non_Dementia', 'Very_Mild_Dementia']


# Function to convert dictionary bytes to image (from your notebook)
def dict_to_image(image_dict):
    if isinstance(image_dict, dict) and 'bytes' in image_dict:
        byte_string = image_dict['bytes']
        # Decode directly using cv2
        nparr = np.frombuffer(byte_string, np.uint8)

        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR) # Read as color
        if img is None:
            raise ValueError("cv2.imdecode failed, image data might be corrupted or in an unsupported format.")
        return img
    else:
        print(f"Skipping row with unexpected image format: {type(image_dict)}")
        return None

# --- Main Script ---
print(f"Reading parquet file: {PARQUET_FILE_PATH}")
df = pd.read_parquet(PARQUET_FILE_PATH, engine="pyarrow")
print(f"Read {len(df)} rows.")

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
print(f"Output folder: {OUTPUT_FOLDER}")

saved_count = {label: 0 for label in range(len(CLASS_NAMES))}

print("Starting image extraction...")
for index, row in df.iterrows():
    label_index = row['label']
    label_name = CLASS_NAMES[label_index]

    if saved_count[label_index] < NUM_IMAGES_PER_CLASS:
        try:
            # Convert the 'image' column data to an image array
            img_array = dict_to_image(row['image'])

            if img_array is not None:
                # Construct the filename
                filename = f"{label_name}_{saved_count[label_index]}.jpg"
                output_path = os.path.join(OUTPUT_FOLDER, filename)

                # Save the image array as a JPG file
                success = cv2.imwrite(output_path, img_array)

                if success:
                    print(f"Saved: {output_path}")
                    saved_count[label_index] += 1
                else:
                    print(f"Failed to save: {output_path}")

        except Exception as e:
            print(f"Error processing row {index}: {e}")

    if all(count >= NUM_IMAGES_PER_CLASS for count in saved_count.values()):
        print("Collected enough images for all classes.")
        break

print("Image extraction finished.")